# 02 - Build the lime-rec table from the `limer` incubation data

[`limer`](https://github.com/cropmodels/limer) (Aramburu Merlos & Hijmans) has lime-incubation data in `inst/data-raw/`. Files are read from GitHub at a pinned commit and written to `data/limer_harmonized.csv` with the columns of notebook 01.

| `kind` | Source | Final pH |
|---|---|---|
| `incubation_measured` | Kamprath (1970), 4 North Carolina soils | measured |
| `incubation_estimated` | Teixeira et al. (2020a), 22 Brazilian soils (limer label `Goncalves2019`) | estimated from Teixeira's regressions (Aramburu Merlos et al. 2023) |
| `recommendation` | Teixeira et al. (2020b): published lime requirement to pH 5.8 and 6.0 for the same 22 soils | — |

Cochrane (1980) and Ananthacumaraswamy (1991) have no final pH and are skipped. `dose_kgCCE_ha` assumes a 15 cm layer at 1.3 g cm⁻³.

In [1]:
import numpy as np, pandas as pd
from pathlib import Path

DATA = Path("../data")
LB_ACRE = {"lb_acre": 1.0, "short_ton_acre": 2000.0, "lb_1000sqft": 43.56}   # unit -> lb per acre
KG_HA_PER_LB_ACRE = 1.12085
BD_DEFAULT = 1.3            # g cm-3, used where a source gives no bulk density
MG_PER_MMOLC = 50.04        # CaCO3: 100.09 g mol-1 / 2 eq
COLS = ["dataset", "kind", "state", "source_name", "soil_id", "ph0", "ph_target", "dpH", "dpH_basis",
        "dose_native", "dose_units", "basis_kind", "cce_factor", "cce_assumed", "mixdepth_cm", "bd", "bd_assumed",
        "dose_kgCCE_ha", "dose_mmolc_kg", "dose_dpH_kgCCE_ha", "dose_dpH_mmolc_kg",
        "above_max_rate", "flag", "ph_note", "note", "source_url"]
soil_kg_ha = lambda depth_cm, bd: depth_cm * bd * 1e5     # mass of the mixing layer

In [2]:
URL = "https://raw.githubusercontent.com/cropmodels/limer/c5ec91f70af879c8ae2b74fa2017e3a1a7e2b6bb/inst/data-raw/"
inc = pd.read_csv(URL + "lime_incubation_studies.csv")
tx = pd.read_csv(URL + "goncalves_soils.csv")
inc.groupby("source").pH.agg(rows="size", with_pH="count")

,rows,with_pH
source,,
Ananthacumaraswamy1991,13,0
Cochrane1980,10,1
Goncalves2019,175,175
Kamprath1970,20,20


In [3]:
SRC = {"Kamprath1970": ("incubation_measured", "https://doi.org/10.2136/sssaj1970.03615995003400020022x"),
       "Goncalves2019": ("incubation_estimated", "https://doi.org/10.36783/18069657rbcs20200078")}
d = inc[inc.source.isin(SRC) & inc.pH.notna()].drop_duplicates(["source", "soil_id", "LR", "pH"])
ph0 = d[d.LR == 0].set_index(["source", "soil_id"]).pH.rename("ph0")
d = d[d.LR > 0].join(ph0, on=["source", "soil_id"])
a = pd.DataFrame({"kind": d.source.map(lambda s: SRC[s][0]), "state": d.source, "source_name": "limer " + d.source,
                  "soil_id": d.soil_id, "ph0": d.ph0, "ph_target": d.pH, "dose_native": d.LR,
                  "note": "OM " + d.OM.round(2).astype(str) + " %", "source_url": d.source.map(lambda s: SRC[s][1])})

t = tx.melt(id_vars=["Soil", "pH", "OM"], value_vars=["LR_pH58", "LR_pH60"], var_name="col", value_name="LR")
b = pd.DataFrame({"kind": "recommendation", "state": "Goncalves2019", "source_name": "Teixeira et al. 2020b (limer goncalves_soils)",
                  "soil_id": t.Soil, "ph0": t.pH, "ph_target": t.col.map({"LR_pH58": 5.8, "LR_pH60": 6.0}), "dose_native": t.LR,
                  "note": "published LR to target; OM " + t.OM.round(2).astype(str) + " %",
                  "source_url": "https://doi.org/10.36783/18069657rbcs20200008"})

lim = pd.concat([a, b], ignore_index=True).assign(
    dataset="limer", dose_units="cmolc_kg", basis_kind="pure_CaCO3", cce_factor=1.0, cce_assumed=False,
    mixdepth_cm=15.0, bd=BD_DEFAULT, bd_assumed=True, dpH_basis="initial_pH", above_max_rate=False,
    ph_note="water pH (Kamprath 1:1; Teixeira 1:2.5)")
lim["dpH"] = lim.ph_target - lim.ph0
lim["dose_mmolc_kg"] = lim.dose_native * 10
lim["dose_kgCCE_ha"] = lim.dose_mmolc_kg * MG_PER_MMOLC * soil_kg_ha(lim.mixdepth_cm, lim.bd) / 1e6
lim["dose_dpH_mmolc_kg"], lim["dose_dpH_kgCCE_ha"] = lim.dose_mmolc_kg, lim.dose_kgCCE_ha
lim["flag"] = np.where(lim.dpH <= 0, "dpH<=0", "")
lim = lim[COLS]
lim.to_csv(DATA / "limer_harmonized.csv", index=False)
lim
# lim.groupby(["state", "kind"]).agg(rows=("dpH", "size"), soils=("soil_id", "nunique"),
#                                    median_mmolc_per_pH=("dose_mmolc_kg", lambda v: (v / lim.loc[v.index, "dpH"]).median().round(1)))

,dataset,kind,state,source_name,soil_id,ph0,ph_target,dpH,dpH_basis,dose_native,...,bd_assumed,dose_kgCCE_ha,dose_mmolc_kg,dose_dpH_kgCCE_ha,dose_dpH_mmolc_kg,above_max_rate,flag,ph_note,note,source_url
0,limer,incubation_measured,Kamprath1970,limer Kamprath1970,dunbar,4.500000,4.9,0.400000,initial_pH,1.165,...,True,1136.7837,11.65,1136.7837,11.65,False,,water pH (Kamprath 1:1; Teixeira 1:2.5),OM 4.0 %,https://doi.org/10.2136/sssaj1970.036159950034...
1,limer,incubation_measured,Kamprath1970,limer Kamprath1970,dunbar,4.500000,5.3,0.800000,initial_pH,2.330,...,True,2273.5674,23.30,2273.5674,23.30,False,,water pH (Kamprath 1:1; Teixeira 1:2.5),OM 4.0 %,https://doi.org/10.2136/sssaj1970.036159950034...
2,limer,incubation_measured,Kamprath1970,limer Kamprath1970,dunbar,4.500000,5.7,1.200000,initial_pH,3.495,...,True,3410.3511,34.95,3410.3511,34.95,False,,water pH (Kamprath 1:1; Teixeira 1:2.5),OM 4.0 %,https://doi.org/10.2136/sssaj1970.036159950034...
3,limer,incubation_measured,Kamprath1970,limer Kamprath1970,dunbar,4.500000,6.0,1.500000,initial_pH,4.660,...,True,4547.1348,46.60,4547.1348,46.60,False,,water pH (Kamprath 1:1; Teixeira 1:2.5),OM 4.0 %,https://doi.org/10.2136/sssaj1970.036159950034...
4,limer,incubation_measured,Kamprath1970,limer Kamprath1970,norfol,4.450000,4.9,0.450000,initial_pH,0.455,...,True,443.9799,4.55,443.9799,4.55,False,,water pH (Kamprath 1:1; Teixeira 1:2.5),OM 2.0 %,https://doi.org/10.2136/sssaj1970.036159950034...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206,limer,recommendation,Goncalves2019,Teixeira et al. 2020b (limer goncalves_soils),LAd1,4.302169,6.0,1.697831,initial_pH,2.540,...,True,2478.4812,25.40,2478.4812,25.40,False,,water pH (Kamprath 1:1; Teixeira 1:2.5),published LR to target; OM 2.79 %,https://doi.org/10.36783/18069657rbcs20200008
207,limer,recommendation,Goncalves2019,Teixeira et al. 2020b (limer goncalves_soils),LVd6,4.600084,6.0,1.399916,initial_pH,1.640,...,True,1600.2792,16.40,1600.2792,16.40,False,,water pH (Kamprath 1:1; Teixeira 1:2.5),published LR to target; OM 1.75 %,https://doi.org/10.36783/18069657rbcs20200008
208,limer,recommendation,Goncalves2019,Teixeira et al. 2020b (limer goncalves_soils),LAd2,4.123819,6.0,1.876181,initial_pH,2.620,...,True,2556.5436,26.20,2556.5436,26.20,False,,water pH (Kamprath 1:1; Teixeira 1:2.5),published LR to target; OM 2.72 %,https://doi.org/10.36783/18069657rbcs20200008
209,limer,recommendation,Goncalves2019,Teixeira et al. 2020b (limer goncalves_soils),RQo1,4.794369,6.0,1.205631,initial_pH,2.390,...,True,2332.1142,23.90,2332.1142,23.90,False,,water pH (Kamprath 1:1; Teixeira 1:2.5),published LR to target; OM 2.19 %,https://doi.org/10.36783/18069657rbcs20200008
